# Crocevia C360 Audience Segmentation Demo

End-to-end Snowpark ML workflow that builds customer segments and product recommendations on top of Crocevia data sources.


### Why This Matters

Meet Alex, the data scientist at a leading retail media network. Alex needs to turn billions of retail transactions into actionable audiences for advertisers in hours, not weeks. This notebook walks Alex end-to-end through Snowflake-powered segmentation, showing how Snowpark ML + Streamlit reduce glue code and keep data securely inside the warehouse.


### Interactive Embeds

Leverage Streamlit inline to preview the recommendation experience directly within Snowsight.


## 1. Session Bootstrap

This notebook is designed to run inside Snowsight. When executed there, `get_active_session()` returns the authenticated Snowpark session. For local testing, populate the optional JSON connection config in the `SNOWFLAKE_CONNECTIONS_C360` environment variable. The run fails fast if the session is missing.


In [ ]:
import json
import os
from datetime import date, timedelta

import modin.pandas as pd
import matplotlib.pyplot as plt
import streamlit as st
import altair as alt
import snowflake.snowpark.modin.plugin
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.exceptions import SnowparkSessionException
from snowflake.snowpark.window import Window
from snowflake.ml.modeling.cluster import KMeans
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.preprocessing import StandardScaler


plt.style.use('seaborn-v0_8')
pd.set_option('display.float_format', lambda v: f"{v:,.2f}")
def resolve_session() -> Session:
    """Return an active Snowpark session or raise a descriptive error."""
    try:
        return get_active_session()
    except SnowparkSessionException:
        pass
    except NameError:
        pass

    config_env = os.environ.get("SNOWFLAKE_CONNECTIONS_C360")
    if not config_env:
        raise RuntimeError("Snowpark session not found. Run in Snowsight or set SNOWFLAKE_CONNECTIONS_C360.")

    connection_parameters = json.loads(config_env)
    return Session.builder.configs(connection_parameters).create()


session = resolve_session()
session.use_database("CROCEVIA_DB")
session.sql("CREATE SCHEMA IF NOT EXISTS CROCEVIA_DB.GOLD_ANALYTICS").collect()
session.use_schema("GOLD_ANALYTICS")

st.write("Session ready for Crocevia C360 demo.")


## 2. Parameters and Guardrails

Configure the analysis window and run key. `write_mode` defaults to append to comply with Crocevia idempotency standards.


In [ ]:
analysis_end_date = date.today()
target_window_days = 365
analysis_start_date = analysis_end_date - timedelta(days=target_window_days - 1)
run_key = analysis_end_date.strftime("%Y%m%d")
write_mode = "append"  # only overwrite with explicit opt-in
st.write(f"Requested analysis window: last {target_window_days} days ending {analysis_end_date}.")


### Demo Flow Highlights

1. **Quality gate** – confirm data completeness with one SQL script.
2. **Feature factory** – build Snowpark ML features directly in the warehouse.
3. **Audience intelligence** – ship curated recommendations into a Streamlit app for merchandisers.


**Prerequisite:** Run `snowflake_sql/check_sales_date_spine.sql` to validate the sales date spine before executing this notebook.


In [ ]:
with st.expander('Data completeness precheck: results'):
    st.write('Run snowflake_sql/check_sales_date_spine.sql prior to this notebook.')


In [ ]:
sales_source = 'CROCEVIA_DB.BRONZE_DATA.CROCEVIA_SALES_20PCT_STORES'
crm_source = 'CROCEVIA_DB.RAW_DATA.CROCEVIA_CRM'
products_source = 'CROCEVIA_DB.BRONZE_DATA.CROCEVIA_PRODUCTS'

sales_df = session.table(sales_source).select('CUSTOMER_ID', 'ORDER_ID', 'SALE_DATE', 'PRODUCT_ID', 'QUANTITY', 'SALES_PRICE_EURO')
crm_df = session.table(crm_source).select(
    'CUSTOMER_ID',
    'FIRST_NAME',
    'LAST_NAME',
    'EMAIL',
    'PHONE',
    'STREET',
    'CITY',
    'POSTAL_CODE',
    'LATITUDE',
    'LONGITUDE',
    'DATE_OF_BIRTH',
    'REGISTRATION_DATE',
    'MARKETING_OPT_IN',
    'OVERLAP_TYPE'
)
products_table = session.table(products_source)
product_columns = [field.name for field in products_table.schema.fields]
name_col = 'PRODUCT_NAME' if 'PRODUCT_NAME' in product_columns else ('PRODUCT' if 'PRODUCT' in product_columns else None)
price_col = 'PRICE_EURO' if 'PRICE_EURO' in product_columns else ('PRICE' if 'PRICE' in product_columns else None)
select_cols = ['PRODUCT_ID'] + ([name_col] if name_col else []) + (['CATEGORY'] if 'CATEGORY' in product_columns else (['PRODUCT_CATEGORY'] if 'PRODUCT_CATEGORY' in product_columns else [])) + (['SUBCATEGORY'] if 'SUBCATEGORY' in product_columns else (['PRODUCT_SUBCATEGORY'] if 'PRODUCT_SUBCATEGORY' in product_columns else [])) + ([price_col] if price_col else []) + (['BRAND'] if 'BRAND' in product_columns else [])
products_df = products_table.select(*select_cols)

st.write('Snowpark sources hydrated: sales, CRM, product catalogs ready for feature engineering.')

## 3. Source Data and Completeness Checks

Load sales, CRM, and product data from Crocevia schemas. Validate that the sales date spine is complete and alert if gaps exist.


In [ ]:
# KPI spotlight keeps compute in Snowflake and surfaces a concise narrative
kpi_window = (analysis_start_date, analysis_end_date)
revenue_metrics = (
    session.table(sales_source)
    .filter((F.col('SALE_DATE') >= F.lit(kpi_window[0])) & (F.col('SALE_DATE') <= F.lit(kpi_window[1])))
    .filter(F.col('CUSTOMER_ID').is_not_null())
    .agg(
        F.sum('SALES_PRICE_EURO').alias('TOTAL_REVENUE'),
        F.count_distinct('CUSTOMER_ID').alias('ACTIVE_CUSTOMERS'),
        F.avg('SALES_PRICE_EURO').alias('AVG_LINE_VALUE')
    )
)
revenue_row = revenue_metrics.collect()[0]
product_count = session.table(products_source).count()
crm_count = session.table(crm_source).count()
kpi_rows = [
    ('Campaign revenue (window)', f"€{revenue_row['TOTAL_REVENUE']:,.0f}"),
    ('Active known customers', f"{revenue_row['ACTIVE_CUSTOMERS']:,.0f}"),
    ('Avg line value', f"€{revenue_row['AVG_LINE_VALUE']:,.2f}"),
    ('Products merchandised', f"{product_count:,}"),
    ('CRM profiles', f"{crm_count:,}"),
]
kpi_frame = session.create_dataframe(kpi_rows, schema=['Metric', 'Value'])
st.dataframe(kpi_frame, use_container_width=True)


### Visual Storyboard

- **Runway Check:** KPI spotlight confirms we’re monetizing the right window before modeling.
- **Feature Lab:** RFM profiles materialize inside Snowflake, ready for segmentation.
- **Activation:** Segment insights feed the Streamlit experience for merchandisers and ad ops.


## 4. Feature Engineering (RFM + Behavioral Attributes)

Aggregate sales into recency, frequency, monetary, and basket-level measures. Join CRM demographics and product diversity metrics.


In [ ]:
rfm_base = (
    sales_df.group_by("CUSTOMER_ID")
    .agg(
        F.max("SALE_DATE").alias("LAST_PURCHASE_DATE"),
        F.sum("SALES_PRICE_EURO").alias("MONETARY_VALUE"),
        F.count_distinct("ORDER_ID").alias("PURCHASE_FREQUENCY"),
        F.sum("QUANTITY").alias("UNITS"),
        F.count_distinct("PRODUCT_ID").alias("PRODUCT_VARIETY"),
    )
    .with_column("RECENCY_DAYS", F.datediff('day', F.col("LAST_PURCHASE_DATE"), F.lit(analysis_end_date)))
)

rfm_enriched = (
    rfm_base.join(crm_df, "CUSTOMER_ID", "left")
    .fillna(
        {
            "MONETARY_VALUE": 0,
            "PURCHASE_FREQUENCY": 0,
            "UNITS": 0,
            "PRODUCT_VARIETY": 0,
        }
    )
)

st.table(rfm_enriched)


## 5. Train Snowpark ML Pipeline

Standardize numeric features and fit a k-means model. Persist both the fitted pipeline and segment assignments.


In [ ]:
feature_columns = [
    'RECENCY_DAYS',
    'PURCHASE_FREQUENCY',
    'MONETARY_VALUE',
    'UNITS',
    'PRODUCT_VARIETY',
]
scaled_columns = [f'SCALED_{col}' for col in feature_columns]

# Explicitly cast DECIMAL/NUMERIC to DOUBLE to avoid implicit conversion warnings
df_for_model = rfm_enriched
for _col in feature_columns:
    df_for_model = df_for_model.with_column(_col, F.col(_col).cast('DOUBLE'))
df_for_model = df_for_model.select(*feature_columns)
training_df = df_for_model.filter(F.col('MONETARY_VALUE') > 0)

# Fit scaler and transform training data
scaler = StandardScaler(input_cols=feature_columns, output_cols=scaled_columns)
scaler_model = scaler.fit(training_df)
scaled_training_df = scaler_model.transform(training_df)

# Fit KMeans on scaled training data
kmeans = KMeans(n_clusters=5, input_cols=scaled_columns, random_state=13)
kmeans_model = kmeans.fit(scaled_training_df)

# Prepare inference DataFrame with proper casts and select id + features
rfm_for_inference = rfm_enriched
for _col in feature_columns:
    rfm_for_inference = rfm_for_inference.with_column(_col, F.col(_col).cast('DOUBLE'))
rfm_for_inference = rfm_for_inference.select('CUSTOMER_ID', *feature_columns)

# Transform full dataset: scale then predict
scaled_all_df = scaler_model.transform(rfm_for_inference)
predictions = kmeans_model.transform(scaled_all_df)

model_version = session.sql("SELECT TO_CHAR(CURRENT_TIMESTAMP(), 'YYYYMMDDHH24MISS')").collect()[0][0]
predictions = predictions.with_column('RUN_KEY', F.lit(run_key))
predictions = predictions.with_column('MODEL_VERSION', F.lit(model_version))

predictions.write.mode(write_mode).save_as_table('CROCEVIA_DB.GOLD_ANALYTICS.C360_CUSTOMER_SEGMENTS')


st.write('Predictions saved to CROCEVIA_DB.GOLD_ANALYTICS.C360_CUSTOMER_SEGMENTS')
st.table(predictions.select('CUSTOMER_ID', 'SEGMENT_LABEL', 'RUN_KEY', 'MODEL_VERSION'))
# Persist learned centroids (scaled space) to GOLD for reuse
centroids = (
    scaled_all_df
    .join(predictions.select('CUSTOMER_ID', 'SEGMENT_LABEL'), 'CUSTOMER_ID', 'inner')
    .group_by('SEGMENT_LABEL')
    .agg(*[F.avg(c).alias(f'CENTROID_{c}') for c in scaled_columns])
)
centroids = centroids.with_column('RUN_KEY', F.lit(run_key)).with_column('MODEL_VERSION', F.lit(model_version))
centroids.write.mode(write_mode).save_as_table('CROCEVIA_DB.GOLD_ANALYTICS.C360_KMEANS_CENTROIDS')
st.write('Centroids saved to CROCEVIA_DB.GOLD_ANALYTICS.C360_KMEANS_CENTROIDS')


In [ ]:
st.markdown('### KPI spotlight')
kpi = (
    predictions
    .join(rfm_enriched.select('CUSTOMER_ID','MONETARY_VALUE','RECENCY_DAYS','PURCHASE_FREQUENCY','LAST_PURCHASE_DATE'), 'CUSTOMER_ID')
)
tot_customers = kpi.select(F.count_distinct('CUSTOMER_ID')).collect()[0][0]
tot_revenue = float(kpi.select(F.sum('MONETARY_VALUE')).collect()[0][0] or 0)
num_segments = kpi.select(F.count_distinct('SEGMENT_LABEL')).collect()[0][0]
active_days = kpi.select(F.count_distinct('LAST_PURCHASE_DATE')).collect()[0][0]
top_seg = (
    kpi.group_by('SEGMENT_LABEL')
    .agg(F.count_distinct('CUSTOMER_ID').alias('CNT'))
    .sort(F.col('CNT').desc()).limit(1).collect()[0]
)
top_seg_id, top_seg_cnt = top_seg[0], top_seg[1]
top_seg_pct = round(100.0 * top_seg_cnt / max(tot_customers, 1), 1)
c1, c2, c3, c4 = st.columns(4)
c1.metric('Customers', f'{tot_customers:,}')
c2.metric('Revenue (€)', f'{tot_revenue:,.0f}')
c3.metric('Active days', f'{active_days:,}')
c4.metric('Segments / Top segment', f'{num_segments} / {top_seg_pct}%')
st.caption(f'Top buyer segment = {top_seg_id}')


In [ ]:
st.markdown('### Audience intelligence')
left, right = st.columns(2)
with left:
    st.subheader('Segment distribution')
    seg_counts = (
        predictions.group_by('SEGMENT_LABEL')
        .agg(F.count_distinct('CUSTOMER_ID').alias('CUSTOMERS'))
        .to_pandas()
    )
    chart = alt.Chart(seg_counts).mark_bar(color='#1f77b4').encode(
        x=alt.X('SEGMENT_LABEL:N', title='Segment'),
        y=alt.Y('CUSTOMERS:Q', title='Customers'),
        tooltip=['SEGMENT_LABEL','CUSTOMERS']
    ).properties(width=480, height=260)
    st.altair_chart(chart, use_container_width=True)
with right:
    st.subheader('Recency vs Frequency (sampled)')
    rf_pd = (
        predictions
        .select('RECENCY_DAYS','PURCHASE_FREQUENCY','SEGMENT_LABEL')
        .limit(5000).to_pandas()
    )
    scatter = alt.Chart(rf_pd).mark_circle(size=28, opacity=0.35).encode(
        x=alt.X('RECENCY_DAYS:Q', title='Recency (days)'),
        y=alt.Y('PURCHASE_FREQUENCY:Q', title='Frequency (# orders)'),
        color=alt.Color('SEGMENT_LABEL:N', legend=None),
        tooltip=['SEGMENT_LABEL','RECENCY_DAYS','PURCHASE_FREQUENCY'],
    ).properties(width=480, height=320)
    st.altair_chart(scatter, use_container_width=True)


## 6. Segment Intelligence and Product Recommendations

**Prerequisite:** Run cell 11 (Train Snowpark ML Pipeline) to define `predictions` before executing this section.

Compute segment-level metrics to identify top buyer segments and generate product recommendations for the Streamlit app.


In [ ]:
segment_metrics = (
    predictions.group_by("SEGMENT_LABEL")
    .agg(
        F.count("CUSTOMER_ID").alias("CUSTOMER_COUNT"),
        F.avg("MONETARY_VALUE").alias("AVG_MONETARY"),
        F.avg("PURCHASE_FREQUENCY").alias("AVG_FREQUENCY"),
        F.avg("RECENCY_DAYS").alias("AVG_RECENCY"),
    )
)
st.table(segment_metrics)

top_segment_row = segment_metrics.sort(F.col("AVG_MONETARY").desc()).limit(1).collect()[0]
top_segment_id = top_segment_row[0]
st.write(f"Top buyer segment: {top_segment_id}")

segment_join_base = (
    predictions.select('CUSTOMER_ID', 'SEGMENT_LABEL')
    .join(sales_df, 'CUSTOMER_ID')
)
product_cols = products_df.schema.names
if 'PRODUCT_ID' in product_cols and ('PRODUCT_NAME' in product_cols):
    segment_join = segment_join_base.join(products_df.select('PRODUCT_ID','PRODUCT_NAME'), 'PRODUCT_ID')
    group_cols = ['SEGMENT_LABEL','PRODUCT_ID','PRODUCT_NAME']
else:
    # Fallback without product catalog
    segment_join = segment_join_base
    group_cols = ['SEGMENT_LABEL','PRODUCT_ID']

segment_recs = (
    segment_join.group_by(*group_cols)
    .agg(F.sum('SALES_PRICE_EURO').alias('REVENUE'))
    .with_column('PRODUCT_RANK', F.rank().over(Window.partition_by('SEGMENT_LABEL').order_by(F.col('REVENUE').desc())))
    .filter(F.col('PRODUCT_RANK') <= 5)
    .with_column('RUN_KEY', F.lit(run_key))
)
segment_recs.write.mode(write_mode).save_as_table('CROCEVIA_DB.GOLD_ANALYTICS.C360_SEGMENT_PRODUCT_RECS')
st.write('Segment product recommendations updated.')
st.table(segment_recs)
st.write("Segment product recommendations updated.")
st.table(segment_recs)


In [ ]:
segment_pd = (
    segment_metrics.select('SEGMENT_LABEL', 'CUSTOMER_COUNT', 'AVG_MONETARY', 'AVG_FREQUENCY')
    .order_by(F.col('AVG_MONETARY').desc())
    .limit(5)
    .to_pandas()
)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(segment_pd['SEGMENT_LABEL'].astype(str), segment_pd['AVG_MONETARY'], color='#1f77b4')
ax.set_title('Top Segments by Average Spend')
ax.set_ylabel('Average monetary value (€)')
ax.set_xlabel('Segment')
for idx, value in enumerate(segment_pd['AVG_MONETARY']):
    ax.text(idx, value + 0.5, f"€{value:.0f}", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
st.table(plt)


In [ ]:
# Inline Streamlit demo of the product recommender
import streamlit.components.v1 as components
streamlit_app_url = 'https://app.snowflake.com/your-account/streamlit/crocevia_c360_recommender'
st.markdown('#### Try the interactive recommender')
st.caption('Launches the Streamlit app within Snowsight for merchandisers.')
components.iframe(streamlit_app_url, height=600)

In [ ]:
top_segment_row = segment_metrics.sort(F.col('AVG_MONETARY').desc()).limit(1).collect()[0]
top_segment_label = top_segment_row['SEGMENT_LABEL']
sample_recs = (
    session.table('CROCEVIA_DB.GOLD_ANALYTICS.C360_SEGMENT_PRODUCT_RECS')
    .filter(F.col('SEGMENT_LABEL') == F.lit(top_segment_label))
    .order_by(F.col('REVENUE').desc())
    .limit(10)
    .select('SEGMENT_LABEL', 'PRODUCT_NAME', 'REVENUE', 'PRODUCT_RANK')
    .to_pandas()
)
sample_recs = sample_recs.rename(columns={'SEGMENT_LABEL': 'Segment', 'PRODUCT_NAME': 'Hero product', 'REVENUE': 'Revenue €', 'PRODUCT_RANK': 'Rank'})
st.dataframe(sample_recs, use_container_width=True)
st.write(f"Segment {top_segment_label} hero products ready for merchandising.")

## 7. Lookalike Audience Scoring

Compute similarity scores between every customer and the top buyer centroid to build a lookalike audience table.


In [ ]:
feature_vectors = predictions.select("CUSTOMER_ID", "SEGMENT_LABEL", *scaled_columns)

centroid_stats = (
    feature_vectors.filter(F.col("SEGMENT_LABEL") == F.lit(top_segment_id))
    .agg(*[F.avg(col).alias(f"CENTROID_{col}") for col in scaled_columns])
)
centroid_row = centroid_stats.collect()[0]
centroid = {col: centroid_row[f"CENTROID_{col}"] for col in scaled_columns}

distance_expr = None
for col in scaled_columns:
    diff = F.col(col) - F.lit(centroid[col])
    term = diff * diff
    distance_expr = term if distance_expr is None else distance_expr + term

lookalike_scores = (
    feature_vectors
    .with_column("DISTANCE_TO_TOP_BUYERS", F.sqrt(distance_expr))
    .with_column("RUN_KEY", F.lit(run_key))
    .with_column("LOOKALIKE_RANK", F.dense_rank().over(Window.order_by(F.col("DISTANCE_TO_TOP_BUYERS"))))
)

lookalike_scores.write.mode(write_mode).save_as_table("CROCEVIA_DB.GOLD_ANALYTICS.C360_LOOKALIKE_SCORES")
st.write("Lookalike scoring complete.")
st.table(lookalike_scores.select("CUSTOMER_ID", "DISTANCE_TO_TOP_BUYERS", "LOOKALIKE_RANK"))


## 8. Audit Logging

Capture run metadata, row counts, and warnings to support operational governance.


In [ ]:
segment_count = predictions.count()
lookalike_count = lookalike_scores.count()

audit_df = session.create_dataframe(
    [
        (
            run_key,
            analysis_start_date,
            analysis_end_date,
            segment_count,
            lookalike_count,
        )
    ],
    schema=[
        "RUN_KEY",
        "START_DATE",
        "END_DATE",
        "SEGMENT_ROWCOUNT",
        "LOOKALIKE_ROWCOUNT",
    ],
)

audit_df.write.mode(write_mode).save_as_table("CROCEVIA_DB.GOLD_ANALYTICS.C360_MODEL_RUN_AUDIT")
st.write("Audit log updated.")
st.table(audit_df)


### Demo Takeaways for Retail Media

- **Faster monetization** – audience KPIs refresh in minutes with Snowpark pipelines.
- **Secure collaboration** – no data extracts; Streamlit surfaces curated insights inside Snowflake.
- **Extensible** – plug in reach/frequency measurement or clean-room activations next.


### Ready to Wow the Advertiser

Run the Streamlit app next and let merchandisers explore curated recommendations while Snowflake keeps everything governed and fast.


## 9. Next Steps

- Publish this notebook to Snowsight and schedule via Tasks or External Access.
- Deploy the Streamlit app under `/crocevia_c360/streamlit/customer_recommender.py`.
- Monitor `CROCEVIA_DB.GOLD_ANALYTICS.C360_MODEL_RUN_AUDIT` for data freshness and anomalies.
